# BiRefNet background removal — test run

Runs **BiRefNet-General** (`ZhengPeng7/BiRefNet`) on a handful of PlantVillage images from GCS and writes:

- `rgba/<class>/<name>.png` — leaf cut out, transparent background
- `mask/<class>/<name>.png` — predicted foreground mask
- `comparison.png` — original | mask | cutout on black, for eyeballing quality

Works on CPU or GPU (fp16 on GPU).

In [ ]:
import shutil, subprocess, sys

# CPU-only torch wheel when there's no GPU (much smaller download).
torch_index = [] if shutil.which("nvidia-smi") else ["--index-url", "https://download.pytorch.org/whl/cpu"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", *torch_index])
# BiRefNet's remote code doesn't load under transformers 5.x. The image's scikit-learn
# predates numpy 2 (which torch pulls in) and breaks transformers' import, so upgrade it too.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.40,<5", "timm", "kornia", "einops", "pillow", "matplotlib", "scikit-learn>=1.5"])

In [ ]:
BUCKET = "gs://leafcheckup-plantvillage"
INPUT_PREFIX = f"{BUCKET}/birefnet_test/input"
OUTPUT_PREFIX = f"{BUCKET}/birefnet_test/output"
LOCAL_IN = "/home/jupyter/birefnet_test/input"
LOCAL_OUT = "/home/jupyter/birefnet_test/output"
MODEL_ID = "ZhengPeng7/BiRefNet"
IMG_SIZE = 1024

In [ ]:
import os

os.makedirs(LOCAL_IN, exist_ok=True)
subprocess.check_call(["gcloud", "storage", "rsync", INPUT_PREFIX, LOCAL_IN, "--recursive"])

In [ ]:
import time
import torch
from torchvision import transforms
from transformers import AutoModelForImageSegmentation

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_float32_matmul_precision("high")

model = AutoModelForImageSegmentation.from_pretrained(MODEL_ID, trust_remote_code=True)
model.to(device).eval()
if device == "cuda":
    model.half()
print("device:", device)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [ ]:
from pathlib import Path
from PIL import Image

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
paths = sorted(p for p in Path(LOCAL_IN).rglob("*") if p.suffix.lower() in IMAGE_EXTS)
print(len(paths), "images")

results = []  # (original, mask, rgba, class name)
timings = []
for p in paths:
    rel = p.relative_to(LOCAL_IN).with_suffix(".png")
    image = Image.open(p).convert("RGB")
    x = transform(image).unsqueeze(0).to(device)
    if device == "cuda":
        x = x.half()

    t0 = time.perf_counter()
    with torch.no_grad():
        pred = model(x)[-1].sigmoid().float().cpu()[0].squeeze()
    timings.append(time.perf_counter() - t0)

    mask = transforms.ToPILImage()(pred).resize(image.size, Image.BILINEAR)
    rgba = image.copy()
    rgba.putalpha(mask)

    for sub, img in (("rgba", rgba), ("mask", mask)):
        out = Path(LOCAL_OUT, sub, rel)
        out.parent.mkdir(parents=True, exist_ok=True)
        img.save(out)
    results.append((image, mask, rgba, p.parent.name))
    print(f"{timings[-1]:6.2f}s  {rel}")

print(f"mean {sum(timings) / len(timings):.2f}s/image (first includes warm-up: {timings[0]:.2f}s)")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(results), 3, figsize=(9, 3 * len(results)))
for row, (image, mask, rgba, cls) in zip(axes, results):
    on_black = Image.new("RGB", image.size, (0, 0, 0))
    on_black.paste(rgba, mask=rgba.split()[-1])
    for ax, img, title in zip(row, (image, mask, on_black), (cls, "mask", "cutout")):
        ax.imshow(img, cmap="gray" if img is mask else None)
        ax.set_title(title, fontsize=8)
        ax.axis("off")
plt.tight_layout()
fig.savefig(Path(LOCAL_OUT, "comparison.png"), dpi=80)
plt.show()

In [ ]:
subprocess.check_call(["gcloud", "storage", "rsync", LOCAL_OUT, OUTPUT_PREFIX, "--recursive"])
print("uploaded to", OUTPUT_PREFIX)